In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [13]:
import librosa
import numpy as np
import os

# Parameters
labels = ["sini bzarba", "ch3al", "tfi", "sini bchwiya"]
DATASET_PATH = "/content/drive/MyDrive/dataset_f"
MAX_LEN = 120  # adjust to match the length used in your training

def extract_features_single(file_path, max_len=120):
    y, sr = librosa.load(file_path, sr=22050)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)  # (40, time_steps)
    mfcc = np.mean(mfcc, axis=0)  # collapse MFCCs → shape: (time_steps,)

    # Pad or truncate to max_len
    if len(mfcc) < max_len:
        mfcc = np.pad(mfcc, (0, max_len - len(mfcc)), mode='constant')
    else:
        mfcc = mfcc[:max_len]

    # Reshape to (batch, height, width, channels)
    mfcc = mfcc.reshape(1, max_len, 1, 1)  # (1, 120, 1, 1)
    return mfcc


# Prediction loop
for label_folder in os.listdir(DATASET_PATH):
    folder_path = os.path.join(DATASET_PATH, label_folder)
    if not os.path.isdir(folder_path):
        continue

    print(f"\n--- Predictions for folder: {label_folder} ---")

    for file in os.listdir(folder_path):
        if file.endswith(".wav"):
            file_path = os.path.join(folder_path, file)
            features = extract_features_single(file_path)
            prediction = model.predict(features)
            predicted_label = labels[np.argmax(prediction)]

            print(f"{file} → Predicted: {predicted_label}")



--- Predictions for folder: sini bzarba ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
WhatsApp Ptt 2025-11-13 at 9.13.27 PM.wav → Predicted: tfi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
WhatsApp Ptt 2025-11-13 at 9.13.33 PM.wav → Predicted: tfi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
WhatsApp Ptt 2025-11-13 at 9.13.43 PM.wav → Predicted: tfi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 110ms/step
WhatsApp Ptt 2025-11-13 at 9.13.40 PM.wav → Predicted: tfi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
WhatsApp Ptt 2025-11-13 at 9.13.40 PM (1).wav → Predicted: tfi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
WhatsApp Ptt 2025-11-13 at 8.29.28 PM.wav → Predicted: tfi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
WhatsApp Ptt 2025-11-13 at 8.55.25 PM.wav → Predicted: tfi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
WhatsApp Ptt 2025-11-13 at 8.29.11 PM.wav → Predicted: tfi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
WhatsApp Ptt 2025-11-13 at 8.03.07 PM.wav → Predicted: tfi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
WhatsApp Ptt 2025-11-13 at 13.13.38.wav

In [10]:
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 118, 1, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 59, 1, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 57, 1, 64)      │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 28, 1, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1792)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       229,504 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 236,358 (923.28 KB)

 Trainable params: 236,356 (923.27 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)